# Notebook 04 | Efficient Frontier and Risk-Aversion Trade-off

Visualize the deterministic Almgren-Chriss frontier implied by the current impact, spread, and volatility assumptions. The frontier should be interpreted as a model based cost risk trade off for schedules, not as a guarantee of realized path performance.

## Frontier Construction

Generate one synthetic market path and sweep the configured risk-aversion grid. The optimizer is solved against the expected market state, which keeps the frontier interpretable and reproducible.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT / 'src'))

from execution_engine.config import load_config
from execution_engine.market.price_process import IntradayMarketSimulator
from execution_engine.optimization.frontier import generate_efficient_frontier
from execution_engine.visualization.plots_frontier import plot_efficient_frontier

config = load_config(PROJECT_ROOT / 'configs' / 'base.yaml')
market = IntradayMarketSimulator(config).simulate(seed=19)
frontier = generate_efficient_frontier(config, market)
frontier

## Cost-Risk Frontier

Each point corresponds to one risk-aversion value. Moving along the curve changes the trade-off between expected execution cost and the penalty assigned to carrying inventory risk.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
plot_efficient_frontier(frontier, ax=ax)
plt.tight_layout()
plt.show()

## Frontier Table

Inspect the optimizer output directly to see how temporary cost, permanent cost, and maximum schedule participation evolve across the risk-aversion grid.

In [ ]:
frontier[['risk_aversion', 'expected_cost', 'risk_cost', 'temporary_cost', 'permanent_cost', 'schedule_max_participation']]

## Interpretation Prompts

- How steeply does expected cost rise as the schedule becomes more risk averse?
- At what point does the optimizer start pressing meaningfully closer to the participation cap?
- Does the deterministic frontier remain informative once execution is evaluated on simulated realized paths?
- Which model inputs would most materially reshape the frontier: spread, vol, or the temporary impact exponent?